# PoRep Market — Happy Path E2E Tests

Interactive notebook for exercising the full deal lifecycle against the **filecoin-boost devnet**.

## Deal lifecycle covered

```
proposeDeal  →  (auto-)acceptDeal  →  ValidatorFactory.create
  →  Validator.createRail  →  Client.transfer (completes deal)
  →  payment settlement (FilecoinPay)  →  terminateDeal
```

### Prerequisites
```
pip install web3 python-dotenv
```

Run the devnet setup scripts first:
```
bash scripts/porep-market/00_setup.sh
bash scripts/porep-market/01_extract_key.sh
bash scripts/porep-market/02_deploy.sh
bash scripts/porep-market/03_deploy_allocator_and_grant_dc.sh
bash scripts/porep-market/04_register_miner.sh
bash scripts/porep-market/05_deploy_token.sh
bash scripts/porep-market/06_setup_sli.sh
```

Contract addresses and keys are read automatically from:
`filecoin-boost/scripts/porep-market/.env`

On devnet all roles (client, provider, PoRep service) share the same deployer key (`PRIVATE_KEY_TEST`).

## 0. Setup

In [ ]:
import builtins as _builtins
import json
import logging
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from web3 import Web3
from web3.exceptions import ContractCustomError
from web3.middleware import ExtraDataToPOAMiddleware

# ── .env ──────────────────────────────────────────────────────────────────────
BOOST_ENV = Path.home() / "Forked/filecoin-boost/scripts/porep-market/.env"
if BOOST_ENV.exists():
    load_dotenv(BOOST_ENV, override=True)
    print(f"Loaded .env from {BOOST_ENV}")
else:
    load_dotenv(override=True)
    print(f"WARNING: {BOOST_ENV} not found")

# ── CONFIG ────────────────────────────────────────────────────────────────────
_key = os.getenv("PRIVATE_KEY_TEST", os.getenv("PRIVATE_KEY", ""))
CONFIG = {
    "rpc_url":                    os.getenv("RPC_URL", "http://127.0.0.1:1234/rpc/v1"),
    "client_key":                 _key,
    "provider_key":               _key,
    "porep_service_key":          _key,
    "porep_market":               os.getenv("POREP_MARKET",      ""),
    "sp_registry":                os.getenv("SP_REGISTRY",       ""),
    "validator_factory":          os.getenv("VALIDATOR_FACTORY", ""),
    "sli_oracle":                 os.getenv("SLI_ORACLE",        ""),
    "sli_scorer":                 os.getenv("SLI_SCORER",        ""),
    "client_sc":                  os.getenv("CLIENT_CONTRACT",   ""),
    "filecoin_pay":               os.getenv("FILECOIN_PAY",      ""),
    "usdfc_token":                os.getenv("USDC_TOKEN",        ""),
    "provider_actor_id":          int(os.getenv("MINER_ACTOR_ID", "1000")),
    # ── deal parameters ───────────────────────────────────────────────────────
    # Devnet providers (04_register_miner.sh) have small capacities:
    #   1000 → 10000 bps / 1000 Mbps / 100 ms / 100% idx / 1 GB
    #   1001 →  8000 bps /  500 Mbps / 200 ms /  80% idx / 5 GB
    #   1002 →  5000 bps /  100 Mbps / 500 ms /  50% idx / 10 GB
    # Use 512 MiB so it fits within miner 1000's 1 GB available capacity.
    "deal_size_bytes":            536870912,   # 512 MiB
    "price_per_sector_per_month": 1_000_000,
    "duration_days":              30,
    "manifest_location":          "ipfs://bafybeigdyrzt5sfp7udm7hu76uh7y26nf3efuylqabf3oclgtqy55fbzdi",
    # Requirements must be met by the provider's registered capabilities:
    "retrievability_bps":         9500,   # miner 1000 has 10000 ✓
    "bandwidth_mbps":             100,    # all miners ✓
    "latency_ms":                 0,      # 0 = no requirement
    "indexing_pct":               0,      # 0 = no requirement
}

# checksum all addresses
_addr_keys = ("porep_market","sp_registry","validator_factory","sli_oracle","sli_scorer","client_sc","filecoin_pay","usdfc_token")
for _k in _addr_keys:
    if CONFIG[_k]:
        CONFIG[_k] = Web3.to_checksum_address(CONFIG[_k])

_missing = [k for k in ("porep_market","sp_registry","validator_factory","sli_oracle","usdfc_token") if not CONFIG[k]]
if _missing:
    print(f"WARNING: missing addresses for {_missing} — run deploy scripts first")
else:
    print("All contract addresses loaded.")

print("\n── CONFIG ──────────────────────────────────────")
for _k, _v in CONFIG.items():
    print(f"  {_k:<30} {_v}")

# ── ABI dir ───────────────────────────────────────────────────────────────────
_nb = globals().get("__vsc_ipynb_file__", "")
ABI_DIR = Path(_nb).parent.parent / "abis" if _nb else BOOST_ENV.parent / "porep-market" / "abis"
assert ABI_DIR.exists(), f"ABI dir not found: {ABI_DIR}"
print(f"\nABI dir : {ABI_DIR}")

def load_abi(name: str):
    with open(ABI_DIR / f"{name}.json") as f:
        return json.load(f)

# ── Logger ────────────────────────────────────────────────────────────────────
_log_dir = Path(_nb).parent / "logs" if _nb else Path.home() / "porep-market-logs"
_log_dir.mkdir(parents=True, exist_ok=True)
if not hasattr(_builtins, "_porep_happy_log"):
    _builtins._porep_happy_log = _log_dir / f"happy-path-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
LOG_FILE = _builtins._porep_happy_log

logger = logging.getLogger("porep.happy")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    _fh = logging.FileHandler(LOG_FILE, mode="a")
    _fh.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-5s  %(message)s"))
    _ch = logging.StreamHandler()
    _ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(_fh)
    logger.addHandler(_ch)
log  = logger.info
logw = logger.warning
loge = logger.error
log(f"=== Session log: {LOG_FILE} ===")

In [ ]:
w3 = Web3(Web3.HTTPProvider(CONFIG["rpc_url"]))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
assert w3.is_connected(), f"Cannot connect to {CONFIG['rpc_url']}"
chain_id = w3.eth.chain_id
log(f"Connected  chain_id={chain_id}  latest_block={w3.eth.block_number}")

In [ ]:
client_acct   = w3.eth.account.from_key(CONFIG["client_key"])        if CONFIG["client_key"]        else None
provider_acct = w3.eth.account.from_key(CONFIG["provider_key"])      if CONFIG["provider_key"]      else None
service_acct  = w3.eth.account.from_key(CONFIG["porep_service_key"]) if CONFIG["porep_service_key"] else None
log(f"Client   : {client_acct.address   if client_acct   else 'NOT SET'}")
log(f"Provider : {provider_acct.address  if provider_acct  else 'NOT SET'}")
log(f"PoRepSvc : {service_acct.address   if service_acct   else 'NOT SET'}")

In [ ]:
porep_market      = w3.eth.contract(address=CONFIG["porep_market"],      abi=load_abi("PoRepMarket"))
sp_registry       = w3.eth.contract(address=CONFIG["sp_registry"],       abi=load_abi("SPRegistry"))
validator_factory = w3.eth.contract(address=CONFIG["validator_factory"], abi=load_abi("ValidatorFactory"))
sli_oracle        = w3.eth.contract(address=CONFIG["sli_oracle"],        abi=load_abi("SLIOracle"))
client_sc         = w3.eth.contract(address=CONFIG["client_sc"],         abi=load_abi("Client")) if CONFIG["client_sc"] else None
log("Contracts loaded.")

In [ ]:
# ── Error map ─────────────────────────────────────────────────────────────────
_error_map: dict[str, str] = {}
for _abi_file in ABI_DIR.glob("*.json"):
    try:
        for _entry in json.load(open(_abi_file)):
            if _entry.get("type") == "error":
                _sig = _entry["name"] + "(" + ",".join(i["type"] for i in _entry.get("inputs", [])) + ")"
                _error_map["0x" + w3.keccak(text=_sig).hex()[:8]] = _sig
    except Exception:
        pass

def decode_custom_error(data: str) -> str:
    return _error_map.get(data[:10], f"unknown error {data[:10]}")

# ── Transaction helper ────────────────────────────────────────────────────────
def send_tx(fn, account, value=0):
    nonce = w3.eth.get_transaction_count(account.address)
    log(f"  → {fn.fn_name}  from={account.address}  nonce={nonce}")
    try:
        tx = fn.build_transaction({
            "from": account.address,
            "nonce": nonce,
            "gasPrice": w3.eth.gas_price,
            "value": value,
            "chainId": chain_id,
        })
    except ContractCustomError as e:
        err = decode_custom_error(e.data)
        loge(f"  ✗ estimate_gas reverted: {err}")
        raise RuntimeError(f"estimate_gas reverted: {err}") from e
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    log(f"  ⏳ sent tx={tx_hash.hex()}")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    if receipt["status"] == 1:
        log(f"  ✅ success  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
    else:
        loge(f"  ❌ reverted  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
        raise RuntimeError(f"Transaction reverted: {tx_hash.hex()}")
    return receipt

def decode_event(contract, receipt, event_name):
    logs = getattr(contract.events, event_name)().process_receipt(receipt)
    return logs[0]["args"] if logs else None

# ── State helpers ─────────────────────────────────────────────────────────────
DEAL_STATES = {0: "Proposed", 1: "Accepted", 2: "Completed", 3: "Rejected", 4: "Terminated"}

# DealProposal is returned as a tuple:
#   p[0]=dealId  p[1]=client  p[2]=provider  p[3]=requirements  p[4]=terms
#   p[5]=validator  p[6]=state  p[7]=railId  p[8]=manifestLocation
# requirements: (retrievabilityBps, bandwidthMbps, latencyMs, indexingPct)
# terms:        (dealSizeBytes, pricePerSectorPerMonth, durationDays)

def fmt_deal(p):
    log(f"  dealId           : {p[0]}")
    log(f"  client           : {p[1]}")
    log(f"  provider (actor) : {p[2]}")
    log(f"  state            : {DEAL_STATES.get(p[6], p[6])}")
    log(f"  validator        : {p[5]}")
    log(f"  railId           : {p[7]}")
    log(f"  manifestLocation : {p[8]}")
    t = p[4]
    log(f"  terms.sizeBytes  : {t[0]:,}")
    log(f"  terms.price/mo   : {t[1]}")
    log(f"  terms.duration   : {t[2]} days")
    r = p[3]
    log(f"  req.retrievability : {r[0]} bps")
    log(f"  req.bandwidth      : {r[1]} Mbps")
    log(f"  req.latency        : {r[2]} ms")
    log(f"  req.indexing       : {r[3]}%")

log(f"Helpers ready. {len(_error_map)} error selectors loaded.")

---
## 1. Preflight checks
Verify connectivity, balances, and that at least one active provider exists in the registry.

In [ ]:
block = w3.eth.get_block("latest")
print(f"Latest block : {block['number']}  (timestamp {block['timestamp']})")

In [ ]:
bal = w3.eth.get_balance(client_acct.address)
log(f"Client FIL balance : {w3.from_wei(bal, 'ether')} FIL")
if bal == 0:
    logw("Client wallet has no FIL — run: docker exec lotus lotus send <address> 1000")

In [ ]:
max_duration = porep_market.functions.MAX_DEAL_DURATION_DAYS().call()
print(f"MAX_DEAL_DURATION_DAYS : {max_duration}")
print("\nPreflight OK ✅")

---
## 2. SP Registry — inspect providers

In [ ]:
try:
    providers = sp_registry.functions.getProviders().call()
except Exception as e:
    providers = []
    logw(f"getProviders() failed: {e}")
log(f"Registered providers ({len(providers)}): {providers}\n")

for actor_id in providers:
    info = sp_registry.functions.getProviderInfo(actor_id).call()
    org, payee, paused, blocked, caps, avail, committed, pending, price = info
    status = "BLOCKED" if blocked else ("PAUSED" if paused else "active")
    log(f"  Provider {actor_id} [{status}]")
    log(f"    org       : {org}")
    log(f"    payee     : {payee}")
    log(f"    available : {avail:,} bytes")
    log(f"    committed : {committed:,} bytes")
    log(f"    pending   : {pending:,} bytes")
    log(f"    price/mo  : {price}")
    log(f"    sli caps  : retrievability={caps[0]} bps  bandwidth={caps[1]} Mbps  latency={caps[2]} ms  indexing={caps[3]}%")

In [ ]:
# Check target provider SLI attestation
actor_id = CONFIG["provider_actor_id"]
attestation = sli_oracle.functions.getAttestation(actor_id).call()
last_update, slis = attestation
print(f"SLI attestation for actor {actor_id}:")
print(f"  lastUpdate (block) : {last_update}")
print(f"  retrievability     : {slis[0]} bps")
print(f"  bandwidth          : {slis[1]} Mbps")
print(f"  latency            : {slis[2]} ms")
print(f"  indexing           : {slis[3]}%")

---
## 3. Propose a deal

`PoRepMarket.proposeDeal(requirements, terms, manifestLocation)` — called by the client wallet.

The registry automatically selects the least-committed matching provider.
If the provider's `pricePerSectorPerMonth` ≤ deal price → the deal **auto-accepts**.

---
## 2.5  Approve MockUSDC spending

The client wallet must approve `PoRepMarket` to pull MockUSDC before `proposeDeal`.
`05_deploy_token.sh` mints 1 M USDC to the deployer — this step just sets the allowance.

In [ ]:
assert client_acct, "Set PRIVATE_KEY_TEST in your environment"
assert CONFIG["usdfc_token"], "USDC_TOKEN not set — run 05_deploy_token.sh first"

USDC_ABI = [
    {"name": "approve",   "type": "function", "inputs": [{"name":"spender","type":"address"},{"name":"amount","type":"uint256"}], "outputs": [{"name":"","type":"bool"}]},
    {"name": "allowance", "type": "function", "inputs": [{"name":"owner","type":"address"},{"name":"spender","type":"address"}], "outputs": [{"name":"","type":"uint256"}]},
    {"name": "balanceOf", "type": "function", "inputs": [{"name":"account","type":"address"}],                                   "outputs": [{"name":"","type":"uint256"}]},
]
usdc_contract = w3.eth.contract(address=Web3.to_checksum_address(CONFIG["usdfc_token"]), abi=USDC_ABI)

bal = usdc_contract.functions.balanceOf(client_acct.address).call()
print(f"Client MockUSDC balance : {bal / 1e6:,.2f} USDC")
assert bal > 0, "Client has no MockUSDC — run 05_deploy_token.sh to mint"

In [ ]:
APPROVE_AMOUNT = 2**256 - 1  # max approval
receipt = send_tx(
    usdc_contract.functions.approve(CONFIG["porep_market"], APPROVE_AMOUNT),
    client_acct,
)

In [ ]:
allowance = usdc_contract.functions.allowance(client_acct.address, CONFIG["porep_market"]).call()
print(f"PoRepMarket allowance : {allowance}")

In [ ]:
assert client_acct, "Set PRIVATE_KEY in your environment"

requirements = (
    CONFIG["retrievability_bps"],
    CONFIG["bandwidth_mbps"],
    CONFIG["latency_ms"],
    CONFIG["indexing_pct"],
)
terms = (
    CONFIG["deal_size_bytes"],
    CONFIG["price_per_sector_per_month"],
    CONFIG["duration_days"],
)

fn = porep_market.functions.proposeDeal(requirements, terms, CONFIG["manifest_location"])
receipt = send_tx(fn, client_acct)

# Extract dealId from DealProposalCreated event
ev = decode_event(porep_market, receipt, "DealProposalCreated")
assert ev, "DealProposalCreated event not found in receipt"

DEAL_ID = ev["dealId"]
print(f"\nDeal proposed  dealId={DEAL_ID}  provider={ev['provider']}")

In [ ]:
# Read back the proposal
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
print("Deal proposal:")
fmt_deal(proposal)

AUTO_ACCEPTED = proposal[6] == 1  # Accepted
print(f"\nAuto-accepted: {AUTO_ACCEPTED}")

---
## 4. Accept the deal  _(skip if auto-accepted)_

The controlling address of the SP actor calls `acceptDeal(dealId)`.

In [ ]:
if AUTO_ACCEPTED:
    print("Deal was auto-accepted — skipping manual acceptDeal step.")
else:
    assert provider_acct, "Set PROVIDER_PRIVATE_KEY to accept the deal manually"
    fn = porep_market.functions.acceptDeal(DEAL_ID)
    receipt = send_tx(fn, provider_acct)
    ev = decode_event(porep_market, receipt, "DealAccepted")
    print(f"Deal accepted  dealId={ev['dealId']}  provider={ev['provider']}")

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
assert proposal[6] == 1, f"Expected Accepted, got {DEAL_STATES[proposal[6]]}"
print(f"State: {DEAL_STATES[proposal[6]]} ✅")

---
## 5. Create a Validator instance

`ValidatorFactory.create(dealId)` deploys a per-deal `BeaconProxy` for the `Validator` contract.  
The new Validator automatically calls `PoRepMarket.updateValidator(dealId)` in its initializer,
so no extra transaction is needed.

In [ ]:
fn = validator_factory.functions.create(DEAL_ID)
receipt = send_tx(fn, client_acct)

In [ ]:
VALIDATOR_ADDR = validator_factory.functions.getInstance(DEAL_ID).call()
print(f"Validator deployed at {VALIDATOR_ADDR}")
validator = w3.eth.contract(address=VALIDATOR_ADDR, abi=load_abi("Validator"))

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
assert proposal["validator"].lower() == VALIDATOR_ADDR.lower(), "Validator address mismatch on deal"
print("PoRepMarket.validator field updated ✅")

---
## Tx 6. Deposit to FilecoinPay + Approve Operator

The client must:
1. **Approve** MockUSDC for FilecoinPay to pull tokens
2. **Deposit** MockUSDC into FilecoinPay (funds the payment rail)
3. **Approve Validator** as operator on FilecoinPay (`setOperatorApproval`)

`maxLockupPeriod` must be ≥ `EPOCHS_IN_MONTH = 86400`.

> **Production:** use `depositWithPermitAndApproveOperator(...)` for a single atomic tx.

In [ ]:
assert CONFIG["filecoin_pay"], "FILECOIN_PAY not set — run 02_deploy.sh first"

FILECOIN_PAY_ABI = [
    {"name": "deposit",            "type": "function", "stateMutability": "payable",
     "inputs": [{"name": "token", "type": "address"}, {"name": "to", "type": "address"}, {"name": "amount", "type": "uint256"}],
     "outputs": []},
    {"name": "withdraw",           "type": "function", "stateMutability": "nonpayable",
     "inputs": [{"name": "token", "type": "address"}, {"name": "amount", "type": "uint256"}],
     "outputs": []},
    {"name": "setOperatorApproval","type": "function", "stateMutability": "nonpayable",
     "inputs": [{"name": "token", "type": "address"}, {"name": "operator", "type": "address"},
                {"name": "approved", "type": "bool"}, {"name": "rateAllowance", "type": "uint256"},
                {"name": "lockupAllowance", "type": "uint256"}, {"name": "maxLockupPeriod", "type": "uint256"}],
     "outputs": []},
    {"name": "operatorApprovals",  "type": "function", "stateMutability": "view",
     "inputs": [{"name": "token", "type": "address"}, {"name": "client", "type": "address"}, {"name": "operator", "type": "address"}],
     "outputs": [{"name": "isApproved", "type": "bool"}, {"name": "rateAllowance", "type": "uint256"},
                 {"name": "lockupAllowance", "type": "uint256"}, {"name": "rateUsage", "type": "uint256"},
                 {"name": "lockupUsage", "type": "uint256"}, {"name": "maxLockupPeriod", "type": "uint256"}]},
    {"name": "accounts",           "type": "function", "stateMutability": "view",
     "inputs": [{"name": "token", "type": "address"}, {"name": "account", "type": "address"}],
     "outputs": [{"name": "funds", "type": "uint256"}, {"name": "lockupCurrent", "type": "uint256"},
                 {"name": "lockupRate", "type": "uint256"}, {"name": "lockupLastSettledAt", "type": "uint256"}]},
    {"name": "settleRail",         "type": "function", "stateMutability": "nonpayable",
     "inputs": [{"name": "railId", "type": "uint256"}, {"name": "epochTo", "type": "uint256"}],
     "outputs": [{"name": "totalSettledAmount", "type": "uint256"}, {"name": "totalNetPayeeAmount", "type": "uint256"},
                 {"name": "totalOperatorCommission", "type": "uint256"}, {"name": "totalNetworkFee", "type": "uint256"},
                 {"name": "finalSettledEpoch", "type": "uint256"}, {"name": "note", "type": "string"}]},
]

filecoin_pay = w3.eth.contract(
    address=Web3.to_checksum_address(CONFIG["filecoin_pay"]),
    abi=FILECOIN_PAY_ABI,
)

usdfc         = Web3.to_checksum_address(CONFIG["usdfc_token"])
validator_addr = Web3.to_checksum_address(VALIDATOR_ADDR)

EPOCHS_IN_MONTH = 86_400
MAX_UINT256     = 2**256 - 1
DEPOSIT_AMOUNT  = 10_000_000  # 10 USDFC (6 decimals) — covers several months of rail payments

# ── Step 1: approve MockUSDC for FilecoinPay ──────────────────────────────────
USDC_ABI = [
    {"name": "approve",   "type": "function", "stateMutability": "nonpayable",
     "inputs": [{"name": "spender", "type": "address"}, {"name": "amount", "type": "uint256"}],
     "outputs": [{"name": "", "type": "bool"}]},
    {"name": "allowance", "type": "function", "stateMutability": "view",
     "inputs": [{"name": "owner", "type": "address"}, {"name": "spender", "type": "address"}],
     "outputs": [{"name": "", "type": "uint256"}]},
    {"name": "balanceOf", "type": "function", "stateMutability": "view",
     "inputs": [{"name": "account", "type": "address"}],
     "outputs": [{"name": "", "type": "uint256"}]},
]
usdc_contract = w3.eth.contract(address=usdfc, abi=USDC_ABI)

bal = usdc_contract.functions.balanceOf(client_acct.address).call()
log(f"Client MockUSDC balance  : {bal / 1e6:,.6f} USDFC")
assert bal >= DEPOSIT_AMOUNT, f"Insufficient MockUSDC — need {DEPOSIT_AMOUNT}, have {bal}"

current_allowance = usdc_contract.functions.allowance(client_acct.address, CONFIG["filecoin_pay"]).call()
if current_allowance < DEPOSIT_AMOUNT:
    send_tx(usdc_contract.functions.approve(CONFIG["filecoin_pay"], MAX_UINT256), client_acct)
    log("MockUSDC approved for FilecoinPay ✅")
else:
    log(f"MockUSDC already approved for FilecoinPay (allowance={current_allowance})")

# ── Step 2: deposit into FilecoinPay ─────────────────────────────────────────
acct_before = filecoin_pay.functions.accounts(usdfc, client_acct.address).call()
log(f"FilecoinPay account before deposit: funds={acct_before[0]}")

send_tx(filecoin_pay.functions.deposit(usdfc, client_acct.address, DEPOSIT_AMOUNT), client_acct)

acct_after = filecoin_pay.functions.accounts(usdfc, client_acct.address).call()
log(f"FilecoinPay account after deposit : funds={acct_after[0]}")

# ── Step 3: approve Validator as operator ────────────────────────────────────
approval = filecoin_pay.functions.operatorApprovals(usdfc, client_acct.address, validator_addr).call()
log(f"operatorApprovals before: isApproved={approval[0]}  maxLockupPeriod={approval[5]}")

if not approval[0] or approval[5] < EPOCHS_IN_MONTH:
    send_tx(
        filecoin_pay.functions.setOperatorApproval(
            usdfc, validator_addr, True,
            MAX_UINT256,      # rateAllowance
            MAX_UINT256,      # lockupAllowance
            EPOCHS_IN_MONTH,  # maxLockupPeriod
        ),
        client_acct,
    )
    approval = filecoin_pay.functions.operatorApprovals(usdfc, client_acct.address, validator_addr).call()
    log(f"operatorApprovals after : isApproved={approval[0]}  maxLockupPeriod={approval[5]} ✅")
else:
    log("Operator already approved ✅")

---
## Tx 7. Create payment rail

`Validator.createRail(token)` — called by the **client** wallet.
Creates a FilecoinPay rail and calls `PoRepMarket.updateRailId(dealId, railId)` internally.

In [ ]:
usdfc = Web3.to_checksum_address(CONFIG["usdfc_token"])
fn = validator.functions.createRail(usdfc)
receipt = send_tx(fn, client_acct)

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
RAIL_ID = proposal["railId"]
assert RAIL_ID != 0, "railId was not set after createRail"
print(f"Rail ID : {RAIL_ID}")

In [ ]:
min_epochs = validator.functions.getMinEpochsBetweenSettlements().call()
print(f"Min epochs between settlements : {min_epochs}  (~{min_epochs / 2880:.1f} days)")

---
## Tx 8. Complete the deal / DDO Allocation

In production the client calls `ClientSC.transfer(params, dealId, dealCompleted=True)` which:
- Submits a DataCap DDO allocation via the Filecoin VerifReg actor
- Stores allocation IDs in the Client SC
- Calls `PoRepMarket.completeDeal()` when `dealCompleted=True`

**Devnet shortcut:** DataCap actors are not available, so we call `PoRepMarket.completeDeal()` directly.
`completeDeal` is restricted to the Client SC address — on devnet the deployer controls that account.

In [ ]:
assert client_sc, "CLIENT_CONTRACT not set — run 02_deploy.sh first"

# pip install cbor2
import cbor2

PROVIDER_ACTOR_ID = CONFIG["provider_actor_id"]
DEAL_SIZE         = CONFIG["deal_size_bytes"]

# ── f0 address of provider: \x00 + varint(actor_id) ─────────────────────────
def _varint(n):
    result = []
    while n >= 0x80:
        result.append((n & 0x7f) | 0x80)
        n >>= 7
    result.append(n)
    return bytes(result)

provider_f0 = b'\x00' + _varint(PROVIDER_ACTOR_ID)

# ── Piece CID: CIDv1, fil-commitment-unsealed (0xf101), sha2-256-trunc254-padded ─
# Format for CBOR tag 42: \x00 (multibase) + CIDv1 + codec varint + mh varint + hash
cid_raw = (
    b'\x00'              # multibase prefix required by CBOR CID encoding
    b'\x01'              # CIDv1
    b'\x81\xe2\x03'      # codec: fil-commitment-unsealed (0xf101)
    b'\x92\x20'          # multihash function: sha2-256-trunc254-padded (0x1012)
    b'\x20'              # hash length: 32 bytes
    + b'\x00' * 32       # hash (dummy zeros — DataCap allocation will be created but not claimable)
)
piece_cid = cbor2.CBORTag(42, cid_raw)  # CBOR tag 42 = CID

# ── Allocation request: 6-element array ──────────────────────────────────────
current_epoch = w3.eth.block_number
allocation = [
    PROVIDER_ACTOR_ID,       # provider actor ID
    piece_cid,               # CID (CBOR tag 42)
    DEAL_SIZE,               # size in bytes (padded piece size)
    518400,                  # term_min (180 days × 2880 epochs/day)
    5_256_000,               # term_max (~5 years)
    current_epoch + 100_000, # expiration of the allocation request
]
operator_data = cbor2.dumps([[allocation], []])  # [allocations, claim_extensions]

# ── BigInt amount = deal size in DataCap bytes ────────────────────────────────
amount_bytes = DEAL_SIZE.to_bytes((DEAL_SIZE.bit_length() + 7) // 8, 'big')

# ── TransferParams ABI tuple ─────────────────────────────────────────────────
# FilAddress{data: bytes}, BigInt{val: bytes, neg: bool}, operator_data: bytes
transfer_params = (
    (provider_f0,),         # to: FilAddress
    (amount_bytes, False),  # amount: BigInt
    operator_data,          # operator_data: bytes (CBOR)
)

log(f"provider_f0    : {provider_f0.hex()}")
log(f"amount         : {DEAL_SIZE:,} bytes")
log(f"operator_data  : {operator_data.hex()}")
log(f"current_epoch  : {current_epoch}")

receipt = send_tx(
    client_sc.functions.transfer(transfer_params, DEAL_ID, True),
    client_acct,
)

ev_logs = client_sc.events.DatacapSpent().process_receipt(receipt)
if ev_logs:
    log(f"DatacapSpent  client={ev_logs[0]['args']['client']}  amount={ev_logs[0]['args']['amount']:,}")

---
## 8. Inspect deal state

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
print("Current deal state:")
fmt_deal(proposal)

In [ ]:
# List all deals in Completed state (ready for payment settlement)
completed = porep_market.functions.getCompletedDeals().call()
print(f"Completed deals ready for settlement: {len(completed)}")
for d in completed:
    print(f"  dealId={d[0]}  provider={d[2]}  railId={d[7]}")

In [ ]:
# SP Registry — check capacity committed vs pending after deal progress
actor_id = CONFIG["provider_actor_id"]
info = sp_registry.functions.getProviderInfo(actor_id).call()
org, payee, paused, blocked, capabilities, avail, committed, pending, price = info
print(f"Provider {actor_id} capacity:")
print(f"  available : {avail:,} bytes")
print(f"  committed : {committed:,} bytes")
print(f"  pending   : {pending:,} bytes")

---
## Tx 10. Modify Rail Payment  _(PoRep service bot)_

After the SP claims DC allocations (Tx 9 — off-chain Filecoin actor call, not shown here),
the bot calls `Validator.modifyRailPayment(railId)` to set the payment rate on the rail.

Requires `POREP_SERVICE_ROLE` — on devnet the deployer holds this role.

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()

if service_acct and proposal[6] == 2:  # Completed
    fn = validator.functions.modifyRailPayment(RAIL_ID)
    receipt = send_tx(fn, service_acct)
    log("Rail payment rate set ✅")
else:
    if not service_acct:
        logw("Skipping — POREP_SERVICE_PRIVATE_KEY not set")
    else:
        logw(f"Skipping — deal not Completed (state={DEAL_STATES[proposal[6]]})")

---
## Tx 11. Settle Rail  _(PoRep service bot — repeating)_

The bot calls `FilecoinPay.settleRail(railId, epochTo)` periodically.
FilecoinPay calls back `Validator.validatePayment(...)` which:
- Checks that ≥ `EPOCHS_IN_MONTH` has passed since last settlement
- Gets SLI score from SLIScorer
- Reads actual deal size from Client SC
- Returns the adjusted payment amount

In [ ]:
if service_acct and proposal[6] == 2:  # Completed
    current_epoch = w3.eth.block_number
    log(f"Settling rail {RAIL_ID} up to epoch {current_epoch} …")
    receipt = send_tx(
        filecoin_pay.functions.settleRail(RAIL_ID, current_epoch),
        service_acct,
    )
    log(f"settleRail ✅  epoch={current_epoch}")

    acct_info = filecoin_pay.functions.accounts(usdfc, client_acct.address).call()
    log(f"Client FilecoinPay : funds={acct_info[0]}  lockup={acct_info[1]}")
else:
    logw(f"Skipping — deal not Completed (state={DEAL_STATES[proposal[6]]})")

---
## Tx 12. Withdraw payments  _(SP / payee)_

The SP calls `FilecoinPay.withdraw(token, amount)` to transfer accumulated earnings to their wallet.

In [ ]:
# On devnet provider == client == deployer, so we withdraw from the same account.
# In production the SP calls this from their own wallet.
sp_acct = provider_acct  # same key on devnet

sp_account = filecoin_pay.functions.accounts(usdfc, sp_acct.address).call()
available = sp_account[0] - sp_account[1]  # funds - lockupCurrent
log(f"SP FilecoinPay account : funds={sp_account[0]}  lockup={sp_account[1]}  available={available}")

if available > 0:
    receipt = send_tx(
        filecoin_pay.functions.withdraw(usdfc, available),
        sp_acct,
    )
    log(f"Withdrawn {available} USDFC units ✅")
else:
    logw("Nothing to withdraw yet (available=0)")

---
## Tx 13. Terminate deal  _(PoRep service bot via Validator)_

`Validator.terminateRail(railId)` — called by holder of `POREP_SERVICE_ROLE`.
- FilecoinPay calls back `Validator.railTerminated(...)` 
- Validator calls `PoRepMarket.terminateDeal(dealId, terminator, endEpoch)`
- PoRepMarket releases SP capacity in SPRegistry

In [ ]:
if service_acct and proposal[6] == 2:  # Completed
    fn = validator.functions.terminateRail(RAIL_ID)
    receipt = send_tx(fn, service_acct)
else:
    print("Skipping — deal must be Completed and POREP_SERVICE_PRIVATE_KEY set.")

In [ ]:
if service_acct and proposal[6] == 2:
    proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
    assert proposal[6] == 4, "Expected state=Terminated"
    print(f"Deal terminated ✅  state={DEAL_STATES[proposal[6]]}")
else:
    print("Skipped.")

---
## 11. Final state summary

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
print("═" * 50)
print("FINAL DEAL STATE")
print("═" * 50)
fmt_deal(proposal)

In [ ]:
remaining_completed = porep_market.functions.getCompletedDeals().call()
print(f"Deals in settlement queue : {len(remaining_completed)}")
for d in remaining_completed:
    print(f"  dealId={d[0]}  provider={d[2]}  railId={d[7]}")

In [ ]:
info = sp_registry.functions.getProviderInfo(CONFIG["provider_actor_id"]).call()
_, _, _, _, _, avail, committed, pending, _ = info
print(f"Provider capacity — avail={avail:,}  committed={committed:,}  pending={pending:,}")
print("\nHappy path run complete.")